# Solution B3 · Many correlated features: OLS, Ridge, regression on the components and PLS

**Type**: Implement with scikit-learn

## Objective

Compare four ways of handling 40 correlated features (generated from 5 latent factors) with only 100 training examples: linear regression, Ridge, **principal component regression** (PCR: PCA + linear regression) with `k` chosen in cross-validation, and **PLS**
(components chosen by looking at `y` too).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline


def generate():
    rng = np.random.default_rng(0)
    n = 200
    L = rng.normal(size=(n, 5))                                       # 5 latent factors
    X = L @ rng.normal(size=(5, 40)) + rng.normal(0, 1.0, (n, 40))    # 40 correlated features
    y = L @ np.array([3, -2, 1, 0.5, 0]) + rng.normal(0, 1.5, n)
    return train_test_split(X, y, test_size=0.5, random_state=0)


X_tr, X_te, y_tr, y_te = generate()
cv = KFold(5, shuffle=True, random_state=0)

## Commented solution

In [ ]:
cv_pcr = pd.Series({k: cross_val_score(make_pipeline(PCA(k), LinearRegression()), X_tr, y_tr, cv=cv, scoring="r2").mean() for k in range(1, 21)})
pcr_k = min(k for k in cv_pcr.index if cv_pcr[k] >= cv_pcr.max() - 0.01)
print(cv_pcr.round(3).to_string())
print("k of the PCR (smallest within 0.01 of the maximum):", pcr_k)

In [ ]:
cv_pls = pd.Series({k: cross_val_score(PLSRegression(k), X_tr, y_tr, cv=cv, scoring="r2").mean() for k in range(1, 11)})
pls_n = min(k for k in cv_pls.index if cv_pls[k] >= cv_pls.max() - 0.01)
pls_has_fewer_components = bool(pls_n < pcr_k)
print(cv_pls.round(3).to_string())
print("PLS components:", pls_n, "| fewer than PCR:", pls_has_fewer_components)

In [ ]:
test_r2 = {
    "OLS": LinearRegression().fit(X_tr, y_tr).score(X_te, y_te),
    "Ridge": RidgeCV(alphas=np.logspace(-2, 4, 20)).fit(X_tr, y_tr).score(X_te, y_te),
    "PCR": make_pipeline(PCA(pcr_k), LinearRegression()).fit(X_tr, y_tr).score(X_te, y_te),
    "PLS": PLSRegression(pls_n).fit(X_tr, y_tr).score(X_te, y_te),
}
pd.Series(test_r2).round(3)

## Check

In [ ]:
with verify():
    check(list(cv_pcr.index) == list(range(1, 21)) and cv_pcr.idxmax() >= 4 and cv_pcr.max() > 0.7, "PCR curve over components 1-20: maximum above 0.7 with at least 4 components", "PCA(k) + LinearRegression")
    check(cv_pcr[1] < 0.3 and 4 <= pcr_k <= 10, "with a single component the R2 is low (below 0.3); the chosen k is between 4 and 10", "smallest k within 0.01 of the maximum")
    check(list(cv_pls.index) == list(range(1, 11)) and pls_n <= 4 and pls_has_fewer_components is True, "PLS: at most 4 components, fewer than PCR", "PLS chooses the directions by looking at y")
    check(set(test_r2) == {"OLS", "Ridge", "PCR", "PLS"} and test_r2["OLS"] < 0.7 and test_r2["Ridge"] > 0.78 and test_r2["PCR"] > 0.78 and test_r2["PLS"] > 0.75, "test R2: OLS below 0.7; Ridge and PCR above 0.78; PLS above 0.75", "final models trained on the training set")
    check(test_r2["PCR"] > test_r2["OLS"] + 0.1, "PCR beats OLS by over 0.1 of R2", "many correlated features and few examples: OLS overfits")
done()

## What you learned

- **OLS suffers from collinearity with few examples:** with 40 features and 100 training examples the test R² is 0.64. The directions with small variance (in the noise) have enormous coefficients that overfit.
- **PCR removes those directions:** keeping only 5 components (the 5 latent factors) the R² goes up to 0.80, on par with Ridge (0.81). With a single component the R² is almost null: the first factor has the largest variance but is not the most useful for `y`.
- **PLS finds the directions by looking at `y`:** with only 2 components it gets 0.78, very close to PCR with 5. In PCR the components are chosen by variance (unsupervised), in PLS by covariance with `y`: more efficient when the signal sits in a few non-dominant directions.
- Ridge, PCR and PLS solve the same problem (collinearity) with different logics: Ridge **shrinks** all the directions in inverse proportion to their variance, PCR **truncates** the low-variance directions, PLS **chooses** the ones useful for the target. In practice Ridge is the first to try: a single hyperparameter, no discrete choices.

## Common pitfall

Assuming that PCR is "the solution to collinearity" without checking that the signal does not sit in the discarded components: C1 shows a case where PCR with few components fails completely.

## Interview question

<details><summary>You have many correlated variables in a regression. What options do you have?</summary>

Ridge (shrinks the coefficients, a single hyperparameter), Lasso or elastic net if I want to select, PCR (regression on the first components) if I think the signal sits in the high-variance directions, PLS if I want target-oriented components.
I compare them in cross-validation with the same split; Ridge is usually the starting point because it is stable and requires no discrete choices.

</details>